# Dentro de la foto · foto → escena 3D para Quest

Todo corre en una GPU gratis de Google. No necesitas PC.

1. Arriba: **Entorno de ejecución → Cambiar tipo de entorno → T4 GPU**.
2. **Entorno de ejecución → Ejecutar todo**.
3. Cuando te lo pida, elige tu foto. Al final te pide permiso de Google Drive y guarda ahí el archivo.
4. En el Quest: abre **drive.google.com** → carpeta *Dentro de la foto* → descarga el `-quest.ply`. Luego abre el visor (https://valibii.github.io/dentro-de-la-foto/) → **Cargar mi .ply** → **Entrar con passthrough**.

Tip de foto: horizontal, bien iluminada, con suelo cerca y fondo lejos.

In [ ]:
#@title 1 · Preparar (la primera vez tarda ~3-4 min)
import os, torch
if not torch.cuda.is_available():
    raise SystemExit("⚠️ No hay GPU. Ve a Entorno de ejecución > Cambiar tipo de entorno > T4 GPU y vuelve a ejecutar.")
print("GPU:", torch.cuda.get_device_name(0))
if not os.path.exists("/content/ml-sharp/pyproject.toml"):
    !git clone -q --depth 1 https://github.com/apple/ml-sharp /content/ml-sharp
    !pip install -q -e /content/ml-sharp
!wget -q -nc -O /content/sharp.pt https://ml-site.cdn-apple.com/models/sharp/sharp_2572gikvuh.pt
print("✅ Listo")

In [ ]:
#@title 2 · Sube tu foto (o varias)
import shutil, pathlib
from google.colab import files
ENTRADA = pathlib.Path("/content/entrada")
shutil.rmtree(ENTRADA, ignore_errors=True); ENTRADA.mkdir()
for nombre, datos in files.upload().items():
    (ENTRADA / nombre.replace(" ", "_")).write_bytes(datos)
print("Fotos:", [p.name for p in ENTRADA.iterdir()])

In [ ]:
#@title 3 · Convertir a 3D
MAX_SPLATS_QUEST = 700000 #@param {type:"integer"}
VIDEO_DE_PRUEBA = False #@param {type:"boolean"}
#@markdown `VIDEO_DE_PRUEBA` hace además un video con movimiento de cámara 3D (la primera vez compila y tarda varios minutos más).
import numpy as np
from plyfile import PlyData, PlyElement
SALIDA = pathlib.Path("/content/salida"); SALIDA.mkdir(exist_ok=True)
extra = "--render" if VIDEO_DE_PRUEBA else ""
!sharp predict -i /content/entrada -o /content/salida -c /content/sharp.pt {extra}

def optimizar(src, dst, max_splats):
    v = PlyData.read(str(src))["vertex"].data
    n0 = len(v)
    alpha = 1 / (1 + np.exp(-v["opacity"]))
    keep = alpha > 0.04
    v, alpha = v[keep], alpha[keep]
    if len(v) > max_splats:
        size = np.exp((v["scale_0"] + v["scale_1"] + v["scale_2"]) / 3)
        idx = np.argpartition(-(alpha * np.sqrt(size)), max_splats)[:max_splats]
        v = v[np.sort(idx)]
    PlyData([PlyElement.describe(v, "vertex")]).write(str(dst))
    print(f"{dst.name}: {n0:,} → {len(v):,} splats, {dst.stat().st_size/1e6:.0f} MB")

for ply in sorted(SALIDA.glob("*.ply")):
    if ply.stem.endswith("-quest"): continue
    optimizar(ply, ply.with_name(ply.stem + "-quest.ply"), MAX_SPLATS_QUEST)

In [ ]:
#@title 4 · Guardar en Google Drive
GUARDAR_EN_DRIVE = True #@param {type:"boolean"}
DESCARGAR_AQUI = False #@param {type:"boolean"}
#@markdown `DESCARGAR_AQUI` baja el archivo al dispositivo donde abriste este notebook (útil si lo abriste desde el navegador del Quest).
resultados = sorted(SALIDA.glob("*-quest.ply")) + sorted(SALIDA.glob("*.mp4"))
if GUARDAR_EN_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    destino = pathlib.Path("/content/drive/MyDrive/Dentro de la foto"); destino.mkdir(parents=True, exist_ok=True)
    for f in resultados:
        shutil.copy(f, destino / f.name); print("Drive ←", f.name)
if DESCARGAR_AQUI:
    for f in resultados: files.download(str(f))
print("✅ Listo. En el Quest: drive.google.com → 'Dentro de la foto' → descarga el -quest.ply → ábrelo en el visor.")